# PolERo: GPU training run (T2.5)

Thin wrapper around `scripts/kaggle_run.py`. All logic lives in the `polero` package. Follow `docs/GPU_RUNBOOK.md`.

**Settings → Accelerator: GPU T4 ×2 · Internet: On** (the model and packages download from public sources; no tokens needed).

In [ ]:
# ---- parameters: edit these ----
CONFIG = "robert-large_mh"     # config name(s) in configs/ without .json; comma-separated runs several,
                               # e.g. "robert-large_mh,xlmr-large_mh,ro-bert-base_mh" (T2.6 screening)
FOLDS = "0,1,2,3,4,5,6"        # every (config, fold) pair is one job; jobs are shared across the GPUs
SMOKE = False                  # True = tiny offline model, ~1 min, to test the setup
INSTALL = "pinned"             # "pinned" | "minimal" (if pinned install fails) | "none"
SRC = None                     # None = auto-detect polero-src.zip under /kaggle/input

In [ ]:
# ---- locate and extract the code package ----
import glob, os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path(os.environ.get("POLERO_WORK", "/kaggle/working"))
def find_src():
    if SRC:
        return Path(SRC)
    hits = glob.glob("/kaggle/input/**/polero-src.zip", recursive=True) or glob.glob("/content/**/polero-src.zip", recursive=True)
    if hits:
        return Path(hits[0])
    dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
    if dirs:
        return dirs[0]  # Kaggle may auto-extract an uploaded zip into a directory
    raise FileNotFoundError("polero-src.zip not found: add the polero-src dataset to this notebook")

src = find_src()
repo = WORK / "polero"
if repo.exists():
    shutil.rmtree(repo)
if src.suffix == ".zip":
    zipfile.ZipFile(src).extractall(repo)
else:
    shutil.copytree(src, repo)
print("code:", src, "->", repo)
print((repo / "VERSION").read_text() if (repo / "VERSION").exists() else "")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("no GPU visible: check Settings -> Accelerator")

In [ ]:
# ---- train (folds split across GPUs), merge OOF, evaluate, zip ----
cmd = [sys.executable, str(repo / "scripts" / "kaggle_run.py"), "--src", str(repo), "--config", CONFIG,
       "--folds", FOLDS, "--install", INSTALL] + (["--smoke"] if SMOKE else [])
subprocess.run(cmd, check=True)
print(sorted(str(p) for p in WORK.glob("*.zip")))